# Notebook 7 — Context + Strong FiLM RoBERTa

This notebook trains the strongest context-aware identity-conditioned model.

Input:
```text
Retrieved Wikimedia context + comment
```

Identity mechanism:
```text
subgroup_id → subgroup embedding → gamma, beta
h_film = gamma(subgroup) * h_cls + beta(subgroup)
```

This combines semantic context with Strong FiLM identity conditioning.


In [1]:
import ast
import json
import random
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, classification_report
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 220)

RANDOM_SEED = 42
MODEL_NAME = "roberta-base"
NUM_LABELS = 3

MAX_LENGTH = 384
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1

EPOCHS = 7
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0

SUBGROUP_DIM = 32
FILM_HIDDEN_DIM = 128
FILM_STRENGTH = 2.0
DROPOUT = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("/home/shayan/Distributional-Hate-Speech-Prediction/data/processed")
CONTEXT_PATH = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/context_artifacts/context_mapped_examples.parquet")

OUTPUT_DIR = Path("context_strong_film_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Device:", DEVICE)
print("Context file:", CONTEXT_PATH)
print("Output directory:", OUTPUT_DIR.resolve())
print("MAX_LENGTH:", MAX_LENGTH)
print("BATCH_SIZE:", BATCH_SIZE)
print("FILM_STRENGTH:", FILM_STRENGTH)


Device: cuda
Context file: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/context_artifacts/context_mapped_examples.parquet
Output directory: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/context_strong_film_outputs
MAX_LENGTH: 384
BATCH_SIZE: 8
FILM_STRENGTH: 2.0


In [2]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


## Load and sanity-check context data

In [3]:
context_df = pd.read_parquet(CONTEXT_PATH)

print("Context dataset:", context_df.shape)
display(context_df.head(2))

required_columns = [
    "comment_id",
    "split",
    "subgroup",
    "text",
    "target_distribution",
    "target_majority_label",
    "context_input_text",
    "retrieved_article_titles",
    "retrieved_summaries",
]

missing = [col for col in required_columns if col not in context_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Rows by split:")
display(context_df["split"].value_counts())

print("Rows by subgroup:")
display(context_df["subgroup"].value_counts())

print("Target majority distribution:")
display(context_df["target_majority_label"].value_counts(normalize=True).sort_index())

print("Unique context inputs:", context_df["context_input_text"].nunique(), "/", len(context_df))
print("Unique comments:", context_df["text"].nunique())

print("\nExample context:")
print("=" * 100)
print(context_df.iloc[0]["context_input_text"])


Context dataset: (9090, 16)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,retrieved_article_titles,retrieved_page_urls,retrieved_similarities,retrieved_summaries,context_input_text,tweet_token_length,context_input_token_length
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[1.0, 0.0, 0.0]",0,0.0,[Left-wing politics],[https://en.wikipedia.org/wiki/Left-wing_politics],[0.14819347858428955],"[Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for the empowerment of marginalized groups. Leftist ideologies vary widely but typically involve a c...",### COMMENT TO CLASSIFY\n\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and ...,60,197
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[0.0, 0.0, 1.0]",2,2.0,[Left-wing politics],[https://en.wikipedia.org/wiki/Left-wing_politics],[0.14819347858428955],"[Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for the empowerment of marginalized groups. Leftist ideologies vary widely but typically involve a c...",### COMMENT TO CLASSIFY\n\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and ...,60,195


Rows by split:


split
train         6360
test          1384
validation    1346
Name: count, dtype: int64

Rows by subgroup:


subgroup
liberal                   2026
neutral                   1512
slightly_liberal          1477
extremely_liberal         1285
slightly_conservative     1079
conservative              1059
extremely_conservative     363
no_opinion                 289
Name: count, dtype: int64

Target majority distribution:


target_majority_label
0    0.734873
1    0.070957
2    0.194169
Name: proportion, dtype: float64

Unique context inputs: 9087 / 9090
Unique comments: 3797

Example context:
### COMMENT TO CLASSIFY
\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill dig him the hole to get there myself

### ANNOTATOR IDEOLOGY
extremely_liberal

### RETRIEVED BACKGROUND
Left-wing politics:
Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for the empowerment of marginalized groups. Leftist ideologies vary widely but typically involve a concern for those disadvantaged by society and a belief in reducing or abolishing unjustified inequalities through radical means. Key concepts include justice, solidarity, cultural pluralism, and freedom from forceful control or coercion. The left is associated with popular or state control of major institutions, trade unions, and critiques of 

In [4]:
def safe_len(x):
    if isinstance(x, list):
        return len(x)
    if isinstance(x, np.ndarray):
        return len(x)
    if isinstance(x, str):
        try:
            return len(ast.literal_eval(x))
        except Exception:
            return np.nan
    return np.nan

print("Retrieved article count per row:")
display(context_df["retrieved_article_titles"].apply(safe_len).value_counts(dropna=False))

print("Retrieved summary count per row:")
display(context_df["retrieved_summaries"].apply(safe_len).value_counts(dropna=False))


Retrieved article count per row:


retrieved_article_titles
1    9090
Name: count, dtype: int64

Retrieved summary count per row:


retrieved_summaries
1    9090
Name: count, dtype: int64

## Subgroup IDs and splits

In [5]:
subgroup_to_id = {
    subgroup: idx
    for idx, subgroup in enumerate(sorted(context_df["subgroup"].unique()))
}

id_to_subgroup = {
    idx: subgroup
    for subgroup, idx in subgroup_to_id.items()
}

context_df["subgroup_id"] = context_df["subgroup"].map(subgroup_to_id)

print("Subgroup mapping:")
print(subgroup_to_id)

train_df = context_df[context_df["split"] == "train"].reset_index(drop=True)
val_df = context_df[context_df["split"].isin(["validation", "val", "dev"])].reset_index(drop=True)
test_df = context_df[context_df["split"] == "test"].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) > 0
assert len(val_df) > 0
assert len(test_df) > 0

display(pd.crosstab(context_df["split"], context_df["subgroup"]))


Subgroup mapping:
{'conservative': 0, 'extremely_conservative': 1, 'extremely_liberal': 2, 'liberal': 3, 'neutral': 4, 'no_opinion': 5, 'slightly_conservative': 6, 'slightly_liberal': 7}
Train: (6360, 17)
Val: (1346, 17)
Test: (1384, 17)


subgroup,conservative,extremely_conservative,extremely_liberal,liberal,neutral,no_opinion,slightly_conservative,slightly_liberal
split,,,,,,,,
test,163,46,197,308,257,46,154,213
train,739,252,907,1407,1027,206,756,1066
validation,157,65,181,311,228,37,169,198


## Token length check

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def count_tokens(text: str) -> int:
    return len(
        tokenizer(
            text,
            truncation=False,
            add_special_tokens=True,
        )["input_ids"]
    )

if "context_input_token_length" not in context_df.columns:
    context_df["context_input_token_length"] = context_df["context_input_text"].apply(count_tokens)

length_summary = {
    "mean": context_df["context_input_token_length"].mean(),
    "median": context_df["context_input_token_length"].median(),
    "p95": context_df["context_input_token_length"].quantile(0.95),
    "max": context_df["context_input_token_length"].max(),
    "pct_over_384": float((context_df["context_input_token_length"] > 384).mean()),
    "pct_over_512": float((context_df["context_input_token_length"] > 512).mean()),
}

display(pd.DataFrame([length_summary]))
print("Rows over MAX_LENGTH:", (context_df["context_input_token_length"] > MAX_LENGTH).sum())


,mean,median,p95,max,pct_over_384,pct_over_512
0,173.806051,168.0,225.0,328,0.0,0.0


Rows over MAX_LENGTH: 0


## Dataset

In [7]:
def parse_distribution(value):
    if isinstance(value, np.ndarray):
        return value.astype(float).tolist()
    if isinstance(value, list):
        return [float(x) for x in value]
    if isinstance(value, str):
        parsed = ast.literal_eval(value)
        return [float(x) for x in parsed]
    raise TypeError(f"Unsupported distribution type: {type(value)}")


class ContextFiLMDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        row = self.dataframe.iloc[idx]

        encoding = self.tokenizer(
            row["context_input_text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "subgroup_id": torch.tensor(row["subgroup_id"], dtype=torch.long),
            "target_distribution": torch.tensor(parse_distribution(row["target_distribution"]), dtype=torch.float),
        }


train_dataset = ContextFiLMDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = ContextFiLMDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = ContextFiLMDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
{k: v.shape for k, v in batch.items()}


{'input_ids': torch.Size([8, 384]),
 'attention_mask': torch.Size([8, 384]),
 'subgroup_id': torch.Size([8]),
 'target_distribution': torch.Size([8, 3])}

## Model

In [8]:
class ContextStrongFiLMRoBERTa(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_subgroups: int,
        subgroup_dim: int = 32,
        film_hidden_dim: int = 128,
        film_strength: float = 2.0,
        num_labels: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.film_strength = film_strength

        self.subgroup_embedding = nn.Embedding(num_subgroups, subgroup_dim)

        self.film_generator = nn.Sequential(
            nn.Linear(subgroup_dim, film_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(film_hidden_dim, hidden_size * 2),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask, subgroup_id):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        subgroup_embedding = self.subgroup_embedding(subgroup_id)

        film_params = self.film_generator(subgroup_embedding)
        gamma, beta = film_params.chunk(2, dim=-1)

        gamma = 1.0 + self.film_strength * torch.tanh(gamma)
        beta = self.film_strength * torch.tanh(beta)

        film_embedding = gamma * cls_embedding + beta

        logits = self.classifier(film_embedding)

        return {
            "logits": logits,
            "log_probs": torch.log_softmax(logits, dim=-1),
            "probs": torch.softmax(logits, dim=-1),
            "gamma": gamma,
            "beta": beta,
        }


model = ContextStrongFiLMRoBERTa(
    model_name=MODEL_NAME,
    num_subgroups=len(subgroup_to_id),
    subgroup_dim=SUBGROUP_DIM,
    film_hidden_dim=FILM_HIDDEN_DIM,
    film_strength=FILM_STRENGTH,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.KLDivLoss(reduction="batchmean")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_update_steps_per_epoch = int(np.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS))
num_training_steps = num_update_steps_per_epoch * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print("Train batches per epoch:", len(train_loader))
print("Training steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Train batches per epoch: 795
Training steps: 5565
Warmup steps: 556


## Metrics

In [9]:
EPS = 1e-12

def kl_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_true = np.clip(y_true, EPS, 1.0)
    y_pred = np.clip(y_pred, EPS, 1.0)
    return np.sum(y_true * np.log(y_true / y_pred), axis=1)

def js_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    return np.array([
        jensenshannon(true_dist, pred_dist, base=2) ** 2
        for true_dist, pred_dist in zip(y_true, y_pred)
    ])

def cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_pred = np.clip(y_pred, EPS, 1.0)
    return -np.sum(y_true * np.log(y_pred), axis=1)

def expected_scores(distributions: np.ndarray) -> np.ndarray:
    labels = np.arange(distributions.shape[1])
    return distributions @ labels

def entropy_values(distributions: np.ndarray) -> np.ndarray:
    return np.array([entropy(dist, base=2) for dist in distributions])

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    true_labels = np.argmax(y_true, axis=1)
    pred_labels = np.argmax(y_pred, axis=1)

    true_expected = expected_scores(y_true)
    pred_expected = expected_scores(y_pred)

    true_entropy = entropy_values(y_true)
    pred_entropy = entropy_values(y_pred)

    metrics = {
        "kl_mean": float(kl_divergence(y_true, y_pred).mean()),
        "js_mean": float(js_divergence(y_true, y_pred).mean()),
        "cross_entropy_mean": float(cross_entropy(y_true, y_pred).mean()),
        "accuracy": float(accuracy_score(true_labels, pred_labels)),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro", zero_division=0)),
        "expected_label_mae": float(mean_absolute_error(true_expected, pred_expected)),
    }

    if len(np.unique(true_entropy)) > 1 and len(np.unique(pred_entropy)) > 1:
        metrics["entropy_pearson"] = float(pearsonr(true_entropy, pred_entropy).statistic)
        metrics["entropy_spearman"] = float(spearmanr(true_entropy, pred_entropy).statistic)
    else:
        metrics["entropy_pearson"] = np.nan
        metrics["entropy_spearman"] = np.nan

    return metrics


## Training

In [10]:
def run_epoch(model, dataloader, optimizer=None, scheduler=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_targets = []
    all_predictions = []

    if is_training:
        optimizer.zero_grad()

    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        subgroup_id = batch["subgroup_id"].to(DEVICE)
        targets = batch["target_distribution"].to(DEVICE)

        with torch.set_grad_enabled(is_training):
            outputs = model(input_ids, attention_mask, subgroup_id)
            raw_loss = criterion(outputs["log_probs"], targets)

            if is_training:
                loss = raw_loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                should_step = (
                    ((step + 1) % GRADIENT_ACCUMULATION_STEPS == 0)
                    or ((step + 1) == len(dataloader))
                )

                if should_step:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    optimizer.zero_grad()
                    if scheduler is not None:
                        scheduler.step()

        total_loss += raw_loss.item() * input_ids.size(0)
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(outputs["probs"].detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)
    y_true = np.vstack(all_targets)
    y_pred = np.vstack(all_predictions)

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(avg_loss)
    return metrics, y_true, y_pred


best_val_kl = float("inf")
best_model_path = OUTPUT_DIR / "context_strong_film_best_model.pt"
history = []

for epoch in range(1, EPOCHS + 1):
    train_metrics, _, _ = run_epoch(model, train_loader, optimizer=optimizer, scheduler=scheduler)
    val_metrics, _, _ = run_epoch(model, val_loader)

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(row)

    print("=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train KL: {train_metrics['kl_mean']:.4f} | Val KL: {val_metrics['kl_mean']:.4f}")
    print(f"Train JS: {train_metrics['js_mean']:.4f} | Val JS: {val_metrics['js_mean']:.4f}")
    print(f"Val Acc: {val_metrics['accuracy']:.4f} | Val Macro F1: {val_metrics['macro_f1']:.4f}")

    if val_metrics["kl_mean"] < best_val_kl:
        best_val_kl = val_metrics["kl_mean"]
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved new best model to {best_model_path}")

history_df = pd.DataFrame(history)
display(history_df)

history_path = OUTPUT_DIR / "context_strong_film_training_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved:", history_path)


Epoch 1/7
Train KL: 0.7361 | Val KL: 0.6214
Train JS: 0.2854 | Val JS: 0.2168
Val Acc: 0.7645 | Val Macro F1: 0.4115
Saved new best model to context_strong_film_outputs/context_strong_film_best_model.pt
Epoch 2/7
Train KL: 0.6013 | Val KL: 0.6347
Train JS: 0.2190 | Val JS: 0.2235
Val Acc: 0.7489 | Val Macro F1: 0.4632
Epoch 3/7
Train KL: 0.5525 | Val KL: 0.5994
Train JS: 0.1944 | Val JS: 0.2118
Val Acc: 0.7845 | Val Macro F1: 0.4657
Saved new best model to context_strong_film_outputs/context_strong_film_best_model.pt
Epoch 4/7
Train KL: 0.5048 | Val KL: 0.6073
Train JS: 0.1797 | Val JS: 0.2065
Val Acc: 0.7808 | Val Macro F1: 0.4533
Epoch 5/7
Train KL: 0.4676 | Val KL: 0.7090
Train JS: 0.1654 | Val JS: 0.2092
Val Acc: 0.7808 | Val Macro F1: 0.4611


KeyboardInterrupt: 

## Test evaluation and predictions

In [11]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

test_metrics, y_true_test, y_pred_test = run_epoch(model, test_loader)

display(test_metrics)

metrics_path = OUTPUT_DIR / "context_strong_film_test_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved:", metrics_path)

predictions_df = test_df.copy()
predictions_df["true_distribution"] = list(y_true_test)
predictions_df["pred_distribution"] = list(y_pred_test)
predictions_df["pred_majority_label"] = np.argmax(y_pred_test, axis=1)
predictions_df["pred_expected_label"] = expected_scores(y_pred_test)
predictions_df["pred_entropy"] = entropy_values(y_pred_test)
predictions_df["kl"] = kl_divergence(y_true_test, y_pred_test)
predictions_df["js"] = js_divergence(y_true_test, y_pred_test)
predictions_df["cross_entropy"] = cross_entropy(y_true_test, y_pred_test)

predictions_path = OUTPUT_DIR / "context_strong_film_test_predictions.parquet"
predictions_df.to_parquet(predictions_path, index=False)
predictions_df.to_csv(OUTPUT_DIR / "context_strong_film_test_predictions.csv", index=False)
print("Saved:", predictions_path)


/tmp/ipykernel_94351/352382768.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))


{'kl_mean': 0.6215674877166748,
 'js_mean': 0.22230612459946236,
 'cross_entropy_mean': 0.6459922790527344,
 'accuracy': 0.7723988439306358,
 'macro_f1': 0.43965479980982297,
 'expected_label_mae': 0.47482133655932685,
 'entropy_pearson': 0.045796314721146335,
 'entropy_spearman': 0.04203762608980165,
 'loss': 0.6215674857140621}

Saved: context_strong_film_outputs/context_strong_film_test_metrics.json
Saved: context_strong_film_outputs/context_strong_film_test_predictions.parquet


## Diagnostics

In [14]:
true_labels = np.argmax(y_true_test, axis=1)
pred_labels = np.argmax(y_pred_test, axis=1)

print("Confusion matrix:")
print(confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2]))

print("\nClassification report:")
print(classification_report(true_labels, pred_labels, labels=[0, 1, 2], zero_division=0))

print("\nPredicted label distribution:")
display(pd.Series(pred_labels).value_counts(normalize=True).sort_index())

print("\nTrue label distribution:")
display(pd.Series(true_labels).value_counts(normalize=True).sort_index())

print("\nAverage predicted distribution:")
print(np.vstack(predictions_df["pred_distribution"].to_numpy()).mean(axis=0))

print("\nMean KL by true majority label:")
display(
    predictions_df
    .groupby("target_majority_label")
    .agg(
        n=("comment_id", "count"),
        mean_kl=("kl", "mean"),
        mean_js=("js", "mean"),
        mean_target_entropy=("target_distribution", lambda x: np.mean([entropy(parse_distribution(v), base=2) for v in x])),
        mean_pred_entropy=("pred_entropy", "mean"),
    )
)

print("Average predicted distribution by true majority label:")
for label in [0, 1, 2]:
    subset = predictions_df[predictions_df["target_majority_label"] == label]
    avg_pred = np.vstack(subset["pred_distribution"].to_numpy()).mean(axis=0)
    print(label, avg_pred)


Confusion matrix:
[[971   0  58]
 [ 77   0  18]
 [162   0  98]]

Classification report:
              precision    recall  f1-score   support

           0       0.80      0.94      0.87      1029
           1       0.00      0.00      0.00        95
           2       0.56      0.38      0.45       260

    accuracy                           0.77      1384
   macro avg       0.46      0.44      0.44      1384
weighted avg       0.70      0.77      0.73      1384


Predicted label distribution:


0    0.874277
2    0.125723
Name: proportion, dtype: float64


True label distribution:


0    0.743497
1    0.068642
2    0.187861
Name: proportion, dtype: float64


Average predicted distribution:
[0.7484423  0.07172151 0.17983672]

Mean KL by true majority label:


,n,mean_kl,mean_js,mean_target_entropy,mean_pred_entropy
target_majority_label,,,,,
0,1029,0.266707,0.113407,0.035754,0.660359
1,95,2.617401,0.782960,0.052632,0.903459
2,260,1.296750,0.448439,0.026836,1.089123


Average predicted distribution by true majority label:
0 [0.816309   0.06339756 0.12029395]
1 [0.6862953  0.08513005 0.22857475]
2 [0.5025541  0.09976561 0.39768004]


In [15]:
print("Performance by subgroup:")

subgroup_rows = []

for subgroup, group in predictions_df.groupby("subgroup"):
    y_true = np.vstack(group["true_distribution"].to_numpy())
    y_pred = np.vstack(group["pred_distribution"].to_numpy())

    row = {
        "subgroup": subgroup,
        "n": len(group),
        **compute_metrics(y_true, y_pred),
    }
    subgroup_rows.append(row)

subgroup_metrics_df = pd.DataFrame(subgroup_rows).sort_values("kl_mean")
display(subgroup_metrics_df)

subgroup_metrics_path = OUTPUT_DIR / "context_strong_film_subgroup_metrics.csv"
subgroup_metrics_df.to_csv(subgroup_metrics_path, index=False)
print("Saved:", subgroup_metrics_path)


Performance by subgroup:


,subgroup,n,kl_mean,js_mean,cross_entropy_mean,accuracy,macro_f1,expected_label_mae,entropy_pearson,entropy_spearman
1,extremely_conservative,46,0.474309,0.194080,0.484854,0.804348,0.501606,0.457664,-0.038698,-0.039300
0,conservative,163,0.509364,0.181057,0.521296,0.815951,0.490287,0.391992,-0.056100,-0.036414
5,no_opinion,46,0.534173,0.193360,0.553093,0.826087,0.532850,0.421583,-0.164464,-0.140309
3,liberal,308,0.584710,0.212368,0.624972,0.795455,0.458092,0.448847,0.101524,0.097052
6,slightly_conservative,154,0.590727,0.217260,0.602286,0.766234,0.436584,0.469857,-0.073448,-0.070746
4,neutral,257,0.672532,0.235745,0.705954,0.766537,0.415226,0.506337,0.086791,0.075093
7,slightly_liberal,213,0.698449,0.245335,0.715594,0.723005,0.397653,0.518578,0.034478,0.027987
2,extremely_liberal,197,0.701319,0.246837,0.722037,0.746193,0.400603,0.515861,0.032162,0.044326


Saved: context_strong_film_outputs/context_strong_film_subgroup_metrics.csv


## Counterfactual subgroup sensitivity

In [16]:
@torch.no_grad()
def predict_distribution(context_text: str, subgroup: str) -> np.ndarray:
    model.eval()

    encoding = tokenizer(
        context_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    subgroup_id = torch.tensor([subgroup_to_id[subgroup]], dtype=torch.long).to(DEVICE)

    outputs = model(input_ids, attention_mask, subgroup_id)
    return outputs["probs"].detach().cpu().numpy()[0]


subgroups = list(subgroup_to_id.keys())

context_lookup = {
    (row["comment_id"], row["subgroup"]): row["context_input_text"]
    for _, row in context_df.iterrows()
}

unique_test_comments = test_df[["comment_id", "text"]].drop_duplicates("comment_id").reset_index(drop=True)

counterfactual_rows = []

for _, row in unique_test_comments.iterrows():
    comment_id = row["comment_id"]

    available_subgroups = [
        subgroup for subgroup in subgroups
        if (comment_id, subgroup) in context_lookup
    ]

    if len(available_subgroups) < 2:
        continue

    preds_by_group = {}
    for subgroup in available_subgroups:
        context_text = context_lookup[(comment_id, subgroup)]
        preds_by_group[subgroup] = predict_distribution(context_text, subgroup)

    pairwise_js = []
    for group_a, group_b in itertools.combinations(available_subgroups, 2):
        js = jensenshannon(preds_by_group[group_a], preds_by_group[group_b], base=2) ** 2
        pairwise_js.append(js)

    counterfactual_rows.append({
        "comment_id": comment_id,
        "text": row["text"],
        "n_available_subgroups": len(available_subgroups),
        "mean_pairwise_js": float(np.mean(pairwise_js)),
        "max_pairwise_js": float(np.max(pairwise_js)),
    })

cf_df = pd.DataFrame(counterfactual_rows)

print("Counterfactual rows:", cf_df.shape)
print("Mean counterfactual JS:", cf_df["mean_pairwise_js"].mean())
print("Median counterfactual JS:", cf_df["mean_pairwise_js"].median())
print("Mean max-pairwise JS:", cf_df["max_pairwise_js"].mean())
print("Median max-pairwise JS:", cf_df["max_pairwise_js"].median())

display(cf_df.sort_values("max_pairwise_js", ascending=False).head(30))

cf_path = OUTPUT_DIR / "context_strong_film_counterfactual_js.csv"
cf_df.to_csv(cf_path, index=False)
print("Saved:", cf_path)


Counterfactual rows: (590, 5)
Mean counterfactual JS: 0.003515531437794484
Median counterfactual JS: 0.0011741579721392388
Mean max-pairwise JS: 0.004257401121175291
Median max-pairwise JS: 0.0014379697853097607


,comment_id,text,n_available_subgroups,mean_pairwise_js,max_pairwise_js
61,8223,"It's funny because the UK has allowed foreign powers to flood their once great country with inbred, low-IQ savages.",4,0.032582,0.080491
530,47662,more attacks on Americans by the traitor in chief all to the glory of stupid white trash,2,0.058376,0.058376
453,44048,"Typical. Go ahead and let them ill3gals in, they finna take over and ran sack this country into a poverty stricken nation.",2,0.055961,0.055961
19,3030,You're a terrorist. Go back to Iraq fucker,2,0.054654,0.054654
39,5453,"I hope this sub closes now because of how much you'd cry. And also the refugees are ruining it, like any good nation. Every time I see a picture of some washed up comic with the title 'Faggot' all of my sperm die.",2,0.044795,0.044795
189,23679,These fucking Indians are getting out of hands. How could they even treat Muslims in front of everyone. :@ May Allah burn them in hell.,2,0.043252,0.043252
359,40210,Canada should kick out khalistani terrorists...they are not sikhs they missusing name of sikhs,3,0.018402,0.034132
483,45185,Shoot a few of those border jumpers and the word will get out. Fuck them. Remember the Alamo!!!,2,0.033778,0.033778
263,31859,"It's like these Miyetti Allah people need a real invasion of their lands to reset their aggressive, greedy brains.",2,0.033705,0.033705
510,46499,y'all are some jackasses and you wanna come to America? Nah. How's about we just bomb your damn home again you fucking roaches,3,0.018956,0.032667


Saved: context_strong_film_outputs/context_strong_film_counterfactual_js.csv


## EL vs EC counterfactual analysis

In [17]:
def pairwise_counterfactual_js(group_a: str, group_b: str) -> pd.DataFrame:
    rows = []
    unique_comments = test_df[["comment_id", "text"]].drop_duplicates("comment_id").reset_index(drop=True)

    for _, row in unique_comments.iterrows():
        comment_id = row["comment_id"]

        if (comment_id, group_a) not in context_lookup:
            continue
        if (comment_id, group_b) not in context_lookup:
            continue

        context_a = context_lookup[(comment_id, group_a)]
        context_b = context_lookup[(comment_id, group_b)]

        pred_a = predict_distribution(context_a, group_a)
        pred_b = predict_distribution(context_b, group_b)

        js = jensenshannon(pred_a, pred_b, base=2) ** 2

        rows.append({
            "comment_id": comment_id,
            "text": row["text"],
            "group_a": group_a,
            "group_b": group_b,
            "js": float(js),
            f"pred_{group_a}": pred_a,
            f"pred_{group_b}": pred_b,
            f"context_{group_a}": context_a,
            f"context_{group_b}": context_b,
        })

    return pd.DataFrame(rows)


if "extremely_liberal" in subgroup_to_id and "extremely_conservative" in subgroup_to_id:
    extreme_df = pairwise_counterfactual_js("extremely_liberal", "extremely_conservative")

    print("EL vs EC rows:", extreme_df.shape)
    print("Extreme liberal vs extreme conservative mean JS:", extreme_df["js"].mean())
    print("Extreme liberal vs extreme conservative median JS:", extreme_df["js"].median())

    display(extreme_df.sort_values("js", ascending=False).head(30))

    extreme_path = OUTPUT_DIR / "context_strong_film_EL_vs_EC_counterfactual.csv"
    extreme_df.to_csv(extreme_path, index=False)
    print("Saved:", extreme_path)


EL vs EC rows: (8, 9)
Extreme liberal vs extreme conservative mean JS: 0.011936295573132992
Extreme liberal vs extreme conservative median JS: 0.010695985330893592


,comment_id,text,group_a,group_b,js,pred_extremely_liberal,pred_extremely_conservative,context_extremely_liberal,context_extremely_conservative
1,20016,Keep the fucking Mexican scum rapists and murderers out of the country and the fucking Arab Islamic terrorists out of the U.S. too!! No Syrian refugees either!! Build a wall and keep all of these fucking parasites ou...,extremely_liberal,extremely_conservative,0.021141,"[0.2028104, 0.08258216, 0.7146075]","[0.10552359, 0.0401975, 0.8542789]",### COMMENT TO CLASSIFY\nKeep the fucking Mexican scum rapists and murderers out of the country and the fucking Arab Islamic terrorists out of the U.S. too!! No Syrian refugees either!! Build a wall and keep all of t...,### COMMENT TO CLASSIFY\nKeep the fucking Mexican scum rapists and murderers out of the country and the fucking Arab Islamic terrorists out of the U.S. too!! No Syrian refugees either!! Build a wall and keep all of t...
5,38441,Indian cunts and dickheads orgasming over developing the occupied and annexed Kashmir so they can settle there while the Kashmiri natives remain locked down in their houses with the Indian occupation forces roaming t...,extremely_liberal,extremely_conservative,0.019772,"[0.48870242, 0.17469715, 0.3366004]","[0.3883188, 0.11503803, 0.49664316]",### COMMENT TO CLASSIFY\nIndian cunts and dickheads orgasming over developing the occupied and annexed Kashmir so they can settle there while the Kashmiri natives remain locked down in their houses with the Indian oc...,### COMMENT TO CLASSIFY\nIndian cunts and dickheads orgasming over developing the occupied and annexed Kashmir so they can settle there while the Kashmiri natives remain locked down in their houses with the Indian oc...
7,49332,Send all these asses back !,extremely_liberal,extremely_conservative,0.016977,"[0.5631097, 0.16088861, 0.27600172]","[0.47034317, 0.11097186, 0.41868502]","### COMMENT TO CLASSIFY\nSend all these asses back !\n\n### ANNOTATOR IDEOLOGY\nextremely_liberal\n\n### RETRIEVED BACKGROUND\nFree migration:\nThe concept of free migration has been debated globally, with various pe...",### COMMENT TO CLASSIFY\nSend all these asses back !\n\n### ANNOTATOR IDEOLOGY\nextremely_conservative\n\n### RETRIEVED BACKGROUND\nOpposition to immigration:\nNational identity plays a significant role in shaping at...
6,43958,"Donald trump is the man , he's right if your not happy in the country you are welcomed to fuck of back to where you came from it's not racist or anything of the sort just clear and simple don't matter who you are fuc...",extremely_liberal,extremely_conservative,0.010801,"[0.5746612, 0.17407703, 0.2512618]","[0.5120896, 0.12820059, 0.3597098]","### COMMENT TO CLASSIFY\nDonald trump is the man , he's right if your not happy in the country you are welcomed to fuck of back to where you came from it's not racist or anything of the sort just clear and simple don...","### COMMENT TO CLASSIFY\nDonald trump is the man , he's right if your not happy in the country you are welcomed to fuck of back to where you came from it's not racist or anything of the sort just clear and simple don..."
0,13695,Yes so every lazy poverty stricken uneducated third world citizen in the entire world from Africa to South America can come here and get welfare and turn America into just as bad of a shithole as their original count...,extremely_liberal,extremely_conservative,0.010591,"[0.6127877, 0.17138977, 0.2158225]","[0.55216587, 0.12864332, 0.31919083]",### COMMENT TO CLASSIFY\nYes so every lazy poverty stricken uneducated third world citizen in the entire world from Africa to South America can come here and get welfare and turn America into just as bad of a shithol...,### COMMENT TO CLASSIFY\nYes so every lazy poverty stricken uneducated third world citizen in the entire world from Africa to South America can come here and get welfare and turn America into just as bad of a shithol...
4,37246,racists b like go back to ur country like ok stop bombing it n ma

Saved: context_strong_film_outputs/context_strong_film_EL_vs_EC_counterfactual.csv


## True EL/EC disagreement vs model disagreement

In [18]:
def valid_dist(x):
    if x is None:
        return False
    try:
        arr = np.array(parse_distribution(x), dtype=float)
    except Exception:
        return False
    if arr.shape[0] != NUM_LABELS:
        return False
    if np.isnan(arr).any():
        return False
    if arr.sum() <= 0:
        return False
    return True


def true_pairwise_disagreement_from_context_df(full_df: pd.DataFrame, group_a: str, group_b: str) -> pd.DataFrame:
    rows = []

    for comment_id, group in full_df.groupby("comment_id"):
        if group_a not in set(group["subgroup"]):
            continue
        if group_b not in set(group["subgroup"]):
            continue

        row_a = group[group["subgroup"] == group_a].iloc[0]
        row_b = group[group["subgroup"] == group_b].iloc[0]

        dist_a = parse_distribution(row_a["target_distribution"])
        dist_b = parse_distribution(row_b["target_distribution"])

        if not valid_dist(dist_a) or not valid_dist(dist_b):
            continue

        dist_a = np.array(dist_a, dtype=float)
        dist_b = np.array(dist_b, dtype=float)
        true_js = jensenshannon(dist_a, dist_b, base=2) ** 2

        rows.append({
            "comment_id": comment_id,
            "text": row_a["text"],
            "true_js": float(true_js),
            f"{group_a}_true_dist": dist_a,
            f"{group_b}_true_dist": dist_b,
            f"{group_a}_true_label": int(np.argmax(dist_a)),
            f"{group_b}_true_label": int(np.argmax(dist_b)),
        })

    return pd.DataFrame(rows)


if "extremely_liberal" in subgroup_to_id and "extremely_conservative" in subgroup_to_id:
    true_df = true_pairwise_disagreement_from_context_df(context_df, "extremely_liberal", "extremely_conservative")

    comparison_df = true_df.merge(
        extreme_df[[
            "comment_id",
            "js",
            "pred_extremely_liberal",
            "pred_extremely_conservative",
        ]],
        on="comment_id",
        how="inner",
    ).rename(columns={"js": "model_js"})

    nonzero = comparison_df[comparison_df["true_js"] > 0].copy()
    nonzero["capture_ratio"] = nonzero["model_js"] / nonzero["true_js"]

    print("N true EL/EC overlap:", len(true_df))
    print("N comparison:", len(comparison_df))
    print("Mean true JS:", comparison_df["true_js"].mean())
    print("Mean model JS:", comparison_df["model_js"].mean())

    if len(nonzero) > 0:
        print("Mean capture ratio, true_js > 0:", nonzero["capture_ratio"].mean())
        print("Median capture ratio, true_js > 0:", nonzero["capture_ratio"].median())

    comparison_df["label_pair"] = (
        comparison_df["extremely_liberal_true_label"].astype(str)
        + "-"
        + comparison_df["extremely_conservative_true_label"].astype(str)
    )

    display(
        comparison_df
        .groupby("label_pair")
        .agg(
            n=("comment_id", "count"),
            mean_true_js=("true_js", "mean"),
            mean_model_js=("model_js", "mean"),
        )
        .sort_values("mean_true_js", ascending=False)
    )

    display(comparison_df.sort_values("true_js", ascending=False).head(30))

    comparison_path = OUTPUT_DIR / "context_strong_film_true_vs_model_EL_EC.csv"
    comparison_df.to_csv(comparison_path, index=False)
    print("Saved:", comparison_path)


N true EL/EC overlap: 101
N comparison: 8
Mean true JS: 0.16448066401642414
Mean model JS: 0.011936295573132992
Mean capture ratio, true_js > 0: 1.5599542263943558
Median capture ratio, true_js > 0: 0.034024470062074166


,n,mean_true_js,mean_model_js
label_pair,,,
0-1,1,1.000000,0.016977
0-0,4,0.077820,0.007499
2-2,3,0.001522,0.016172


,comment_id,text,true_js,extremely_liberal_true_dist,extremely_conservative_true_dist,extremely_liberal_true_label,extremely_conservative_true_label,model_js,pred_extremely_liberal,pred_extremely_conservative,label_pair
7,49332,Send all these asses back !,1.000000,"[1.0, 0.0, 0.0]","[0.0, 1.0, 0.0]",0,1,0.016977,"[0.5631097, 0.16088861, 0.27600172]","[0.47034317, 0.11097186, 0.41868502]",0-1
0,13695,Yes so every lazy poverty stricken uneducated third world citizen in the entire world from Africa to South America can come here and get welfare and turn America into just as bad of a shithole as their original count...,0.311278,"[0.5, 0.0, 0.5]","[1.0, 0.0, 0.0]",0,0,0.010591,"[0.6127877, 0.17138977, 0.2158225]","[0.55216587, 0.12864332, 0.31919083]",0-0
1,20016,Keep the fucking Mexican scum rapists and murderers out of the country and the fucking Arab Islamic terrorists out of the U.S. too!! No Syrian refugees either!! Build a wall and keep all of these fucking parasites ou...,0.004567,"[0.08421052992343903, 0.10526315867900848, 0.8105263113975525]","[0.06666667014360428, 0.06666667014360428, 0.8666666746139526]",2,2,0.021141,"[0.2028104, 0.08258216, 0.7146075]","[0.10552359, 0.0401975, 0.8542789]",2-2
2,20033,"STOP SUCKING THAT FATASS' DICK, YOU NIGGERLOVING KIKE",0.000000,"[0.0, 0.0, 1.0]","[0.0, 0.0, 1.0]",2,2,0.007604,"[0.11537573, 0.057555087, 0.82706916]","[0.06870741, 0.033807386, 0.89748514]",2-2
3,20062,I've always been proud of the Igbo tribe and our culture. Yesterday made me even more proud. You people are too fine abeg. OgbakoUmuIgboDay,0.000000,"[1.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0,0,0.000744,"[0.9724026, 0.019473098, 0.008124386]","[0.9632639, 0.022191118, 0.014544969]",0-0
4,37246,racists b like go back to ur country like ok stop bombing it n maybe i will,0.000000,"[1.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0,0,0.007860,"[0.54402477, 0.18472187, 0.27125338]","[0.5180952, 0.12822384, 0.35368094]",0-0
5,38441,Indian cunts and dickheads orgasming over developing the occupied and annexed Kashmir so they can settle there while the Kashmiri natives remain locked down in their houses with the Indian occupation forces roaming t...,0.000000,"[0.0, 0.0, 1.0]","[0.0, 0.0, 1.0]",2,2,0.019772,"[0.48870242, 0.17469715, 0.3366004]","[0.3883188, 0.11503803, 0.49664316]",2-2
6,43958,"Donald trump is the man , he's right if your not happy in the country you are welcomed to fuck of back to where you came from it's not racist or anything of the sort just clear and simple don't matter who you are fuc...",0.000000,"[1.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0,0,0.010801,"[0.5746612, 0.17407703, 0.2512618]","[0.5120896, 0.12820059, 0.3597098]",0-0


Saved: context_strong_film_outputs/context_strong_film_true_vs_model_EL_EC.csv


## Inspect highest EL/EC context effect

In [19]:
if "extreme_df" in globals() and len(extreme_df) > 0:
    top_row = extreme_df.sort_values("js", ascending=False).iloc[0]

    print("Comment:")
    print(top_row["text"])

    print("\nPrediction EL:")
    print(top_row["pred_extremely_liberal"])

    print("\nPrediction EC:")
    print(top_row["pred_extremely_conservative"])

    print("\nContext EL:")
    print(top_row["context_extremely_liberal"])

    print("\nContext EC:")
    print(top_row["context_extremely_conservative"])


Comment:
Keep the fucking Mexican scum rapists and murderers out of the country and the fucking Arab Islamic terrorists out of the U.S. too!! No Syrian refugees either!! Build a wall and keep all of these fucking parasites out of the United States. We'll round up the ones here illegally and ship them the fuck out!!!!

Prediction EL:
[0.2028104  0.08258216 0.7146075 ]

Prediction EC:
[0.10552359 0.0401975  0.8542789 ]

Context EL:
### COMMENT TO CLASSIFY
Keep the fucking Mexican scum rapists and murderers out of the country and the fucking Arab Islamic terrorists out of the U.S. too!! No Syrian refugees either!! Build a wall and keep all of these fucking parasites out of the United States. We'll round up the ones here illegally and ship them the fuck out!!!!

### ANNOTATOR IDEOLOGY
extremely_liberal

### RETRIEVED BACKGROUND
Free migration:
The concept of free migration has been debated globally, with various perspectives on its moral, economic, and cultural implications. Some argue tha

## Interpretation Guide

Compare this model against:

1. Embedding baseline, no context
2. Strong FiLM, no context
3. Context + Embedding
4. Context + Strong FiLM

Most important metrics:

```text
KL / JS
Macro F1
Class-2 recall
Mean counterfactual JS
EL↔EC model JS
True-vs-model capture ratio
```
